In [1]:
from langgraph.graph import StateGraph , START, END
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.output_parsers import PydanticOutputParser
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.checkpoint.memory import MemorySaver


In [2]:
load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model = ChatHuggingFace(llm = llm)


In [4]:
class Chatstate(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [5]:
def chat_node(state:Chatstate):
    messages = state['messages']

    respond = model.invoke(messages)

    return {
        'messages' : [respond]
    }
    

In [6]:
pointer = MemorySaver()

graph = StateGraph(Chatstate)

graph.add_node('chat_node' , chat_node)
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node' ,  END)

app = graph.compile(checkpointer= pointer)

In [14]:
thread_id = "1"
while True:
    user_message = input("Type Here")
    print("user" , user_message)

    if user_message.strip().lower() in ['exit' , 'quit'] :
        break

    config = {
        'configurable' : {'thread_id': thread_id}
    }
    result = app.invoke ({'messages' : [HumanMessage(content=user_message)]}, config= config)
    
    print("AI:" , result['messages'][-1].content)
         
             
            

user hi
AI: You're saying hi again. Would you like to discuss something or is it just a casual hello?
user add 5 + 5
AI: The result of 5 + 5 is 10.
user multiply the result by 0
AI: Multiplying 10 by 0 results in 0.
user exit
